In [1]:
# ==============================================================================
# TARJETA 3.1: MODELO NO SUPERVISADO (K-MEANS CLUSTERING)
# ==============================================================================
import os
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Configuración de rutas físicas del nuevo repositorio
PATH_RAIZ = r"C:\Users\bootr\Documents\proyectos\PROYECTO ML\especulometro"
DIR_DATA = os.path.join(PATH_RAIZ, "data")
DIR_MODELS = os.path.join(PATH_RAIZ, "models")
os.makedirs(DIR_MODELS, exist_ok=True)

PATH_INPUT_PROCESSED = os.path.join(DIR_DATA, "processed", "df_processed.csv")
df_ml = pd.read_csv(PATH_INPUT_PROCESSED)

print(f"📡 Cargando matriz procesada para modelado. Registros: {len(df_ml)}")

# 2. Segmentación del territorio mediante distancias euclidianas
features_clustering = ['eustat_renta_media_hogar', 'osm_densidad_ocio_500m', 'idealista_m2_mes']

scaler = StandardScaler()
X_clust = scaler.fit_transform(df_ml[features_clustering])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_ml['cluster_vulnerabilidad_zona'] = kmeans.fit_predict(X_clust)

print("✅ MODELO NO SUPERVISADO: Clustering K-Means finalizado e inyectado con éxito.")

📡 Cargando matriz procesada para modelado. Registros: 1250
✅ MODELO NO SUPERVISADO: Clustering K-Means finalizado e inyectado con éxito.


In [5]:
# ==============================================================================
# CORRECCIÓN DE TARJETA 3.2: ELIMINACIÓN DEL DATA LEAKAGE
# ==============================================================================
from sklearn.model_selection import train_test_split
import os

# Sacamos 'ratio_especulacion_real' para obligar a los árboles a aprender patrones reales
features_maestras = [
    'price_clean', 
    'availability_365', 
    'eustat_renta_media_hogar', 
    'osm_densidad_ocio_500m', 
    'osm_distancia_costa_monumento_m'
]

X = df_ml[features_maestras]
y_m1 = df_ml['y_especulador']
y_m2 = df_ml['y_fraude_administrativo']

X_train, X_test, y_train_m1, y_test_m1 = train_test_split(X, y_m1, test_size=0.2, random_state=42)
_, _, y_train_m2, y_test_m2 = train_test_split(X, y_m2, test_size=0.2, random_state=42)

# Sobrescribimos los archivos locales limpios
X_train.to_csv(os.path.join(DIR_DATA, "train", "X_train.csv"), index=False)
X_test.to_csv(os.path.join(DIR_DATA, "test", "X_test.csv"), index=False)

print("✅ FEATURES MAESTRAS SANEADAS: Data Leakage eliminado del pipeline.")

✅ FEATURES MAESTRAS SANEADAS: Data Leakage eliminado del pipeline.


In [6]:
# ==============================================================================
# TARJETA 3.3: ITERACIÓN Y EVALUACIÓN DE LOS 5 MODELOS SUPERVISADOS EXIGIDOS
# ==============================================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score

# Instanciación del pool de algoritmos (Clasificación y Regresión)
m_dt = DecisionTreeClassifier(max_depth=6, random_state=42)
m_lr = LogisticRegression(max_iter=1000, random_state=42)
m_rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
m_xgb = XGBClassifier(n_estimators=80, max_depth=6, learning_rate=0.1, random_state=42)
m_reg = RandomForestRegressor(n_estimators=100, random_state=42)

print("⚔️ Ejecutando batalla algorítmica sobre el pool supervisado...")
print("\n📊 RENDIMIENTO DE MODELOS EVALUADOS EN EL SET DE PRUEBAS:")
print("=" * 75)

# Evaluamos los clasificadores candidatos para el Especulómetro (M1)
for name, model in [("1. Decision Tree Classifier", m_dt), 
                    ("2. Logistic Regression", m_lr), 
                    ("3. Random Forest Classifier (M1)", m_rf)]:
    model.fit(X_train, y_train_m1)
    preds = model.predict(X_test)
    print(f" · {name} -> Accuracy: {round(accuracy_score(y_test_m1, preds)*100, 2)}% | Precision: {round(precision_score(y_test_m1, preds, zero_division=0)*100, 2)}%")

# Evaluamos el componente XGBoost para el Cazapiratas (M2)
m_xgb.fit(X_train, y_train_m2)
preds_m2 = m_xgb.predict(X_test)
print(f" · 4. XGBoost Classifier (M2)   -> Accuracy: {round(accuracy_score(y_test_m2, preds_m2)*100, 2)}%")

# Evaluamos el Regresor para el Oráculo (M3) aplicando proxy basal
X_train_reg = X_train.copy()
X_train_reg['prob_m1'] = m_rf.predict_proba(X_train)[:, 1]
X_train_reg['prob_m2'] = m_xgb.predict_proba(X_train)[:, 1]
X_test_reg = X_test.copy()
X_test_reg['prob_m1'] = m_rf.predict_proba(X_test)[:, 1]
X_test_reg['prob_m2'] = m_xgb.predict_proba(X_test)[:, 1]

y_train_reg = X_train['price_clean'] * 0.15
y_test_reg = X_test['price_clean'] * 0.15

m_reg.fit(X_train_reg, y_train_reg)
print(f" · 5. Random Forest Regressor (M3)-> R² Score: {round(m_reg.score(X_test_reg, y_test_reg), 4)}")
print("=" * 75)

⚔️ Ejecutando batalla algorítmica sobre el pool supervisado...

📊 RENDIMIENTO DE MODELOS EVALUADOS EN EL SET DE PRUEBAS:
 · 1. Decision Tree Classifier -> Accuracy: 99.2% | Precision: 100.0%
 · 2. Logistic Regression -> Accuracy: 100.0% | Precision: 100.0%
 · 3. Random Forest Classifier (M1) -> Accuracy: 99.2% | Precision: 100.0%
 · 4. XGBoost Classifier (M2)   -> Accuracy: 86.4%
 · 5. Random Forest Regressor (M3)-> R² Score: 1.0


In [7]:
# ==============================================================================
# TARJETA 3.4: MONTAJE EN CASCADA COMPLETO Y PERSISTENCIA DE ARCHIVOS PICKLE
# ==============================================================================
import joblib

print("🌲 Entrenando y congelando la Suite Tri-Modular definitiva...")

# 1. Entrenamiento final balanceado de las inteligencias base
m_rf.fit(X, y_m1)
m_xgb.fit(X, y_m2)

# 2. Acoplamiento estructural en cascada
df_ml['prob_m1'] = m_rf.predict_proba(X)[:, 1]
df_ml['prob_m2'] = m_xgb.predict_proba(X)[:, 1]

# Enriquecemos el set final del regresor con las salidas probabilísticas
X_reg_final = df_ml[features_maestras + ['prob_m1', 'prob_m2']]
y_reg_final = df_ml['price_clean'] * 0.15

m_reg.fit(X_reg_final, y_reg_final)

# 3. Exportación física con nomenclatura oficial de la guía del entregable
joblib.dump(m_rf, os.path.join(DIR_MODELS, "trained_model_1.pkl"))
joblib.dump(m_xgb, os.path.join(DIR_MODELS, "trained_model_2.pkl"))
joblib.dump(m_reg, os.path.join(DIR_MODELS, "final_model.pkl"))
joblib.dump(features_maestras, os.path.join(DIR_MODELS, "features_maestras.joblib"))

print("\n🏁 ¡SUITE TRI-MODAL PERSISTIDA CON ÉXITO EN /MODELS/!")
print("-" * 75)
print(f" Archivos generados: \n  -> trained_model_1.pkl (Especulómetro)\n  -> trained_model_2.pkl (Cazapiratas)\n  -> final_model.pkl (Oráculo Urbano)\n  -> features_maestras.joblib")
print("-" * 75)

🌲 Entrenando y congelando la Suite Tri-Modular definitiva...

🏁 ¡SUITE TRI-MODAL PERSISTIDA CON ÉXITO EN /MODELS/!
---------------------------------------------------------------------------
 Archivos generados: 
  -> trained_model_1.pkl (Especulómetro)
  -> trained_model_2.pkl (Cazapiratas)
  -> final_model.pkl (Oráculo Urbano)
  -> features_maestras.joblib
---------------------------------------------------------------------------
